# Image Reconstruction Options

This notebook is a test bed to run and check all implemented image reconstruction methods.

In [ ]:
# This cells setups the environment when executed in Google Colab.
try:
    import google.colab
    !curl -s https://raw.githubusercontent.com/ibs-lab/cedalion/dev/scripts/colab_setup.py -o colab_setup.py
    # Select branch with --branch "branch name" (default is "dev")
    %run colab_setup.py
except ImportError:
    pass

## Notebook configuration 
Decide for an example dataset with a sparse probe or a high-density probe for DOT. 
The notebook will load example data accordingly.

Also specify, if precomputed results of the photon propagation should be used and
if the 3D visualizations should be interactive.

In [ ]:
DATASET = "fingertappingDOT" # high-density montage
HEAD_MODEL = "colin27"
FORWARD_MODEL = "MCX" # photon monte carlo
PRECOMPUTED_FLUENCE = True

# set this flag to True to enable interactive 3D plots
INTERACTIVE_PLOTS = False

In [ ]:
import pyvista as pv

if INTERACTIVE_PLOTS:
    pv.set_jupyter_backend("server")
else:
    pv.set_jupyter_backend("static")

import itertools
import traceback
from pathlib import Path
from tempfile import TemporaryDirectory

import xarray as xr
from IPython.display import Image

import cedalion

import cedalion.dot as dot
import cedalion.sigproc.motion_correct as motion_correct
import cedalion.xrutils as xrutils
from cedalion import units
from cedalion.io.forward_model import load_Adot
from cedalion.sigproc.quality import measurement_variance
from cedalion.plots import image_recon_multi_view

# fail on unit errors
xrutils.unit_stripping_is_error()

xr.set_options(display_expand_data=False);

In [ ]:
# helper function to display gifs in rendered notbooks
def display_image(fname : str):
    display(Image(data=open(fname,'rb').read(), format='png'))

## Working Directory
In this notebook the output of the fluence and sensitivity calculations are stored in a temporary directory. This will be 
deleted when the notebook ends.

In [ ]:
temporary_directory = TemporaryDirectory()
tmp_dir_path = Path(temporary_directory.name)

## Load a finger-tapping dataset 

In [ ]:
rec = cedalion.datasets.get_fingertappingDOT()

In [ ]:
geo3d_meas = rec.geo3d
display(geo3d_meas)

The measurement list is a `pandas.DataFrame` that describes which source-detector pairs form channels.

In [ ]:
meas_list = rec._measurement_lists["amp"]
display(meas_list.head(5))

Event/stimulus information is also stored in a `pandas.DataFrame`. 

In [ ]:
rec.stim

For clarity, events are given more descriptive names:

In [ ]:
rec.stim.cd.rename_events(
    {
        "1": "Control",
        "2": "FTapping/Left",
        "3": "FTapping/Right",
        "4": "BallSqueezing/Left",
        "5": "BallSqueezing/Right",
    }
)

# count number of trials per trial_type
display(
    rec.stim.groupby("trial_type")[["onset"]]
    .count()
    .rename({"onset": "#trials"}, axis=1)
)

## Preprocessing
Perform motion correction, conversion to optical density and bandpass filtering.

In [ ]:
rec["od"] = cedalion.nirs.int2od(rec["amp"])
rec["od_tddr"] = motion_correct.tddr(rec["od"])
rec["od_wavelet"] = motion_correct.wavelet(rec["od_tddr"])

# bandpass filter the data
rec["od_freqfiltered"] = rec["od_wavelet"].cd.freq_filter(
    fmin=0.01, fmax=0.5, butter_order=4
)

## Calculate block averages in optical density

In [ ]:
# segment data into epochs
epochs = rec["od_freqfiltered"].cd.to_epochs(
    rec.stim,  # stimulus dataframe
    ["FTapping/Left", "FTapping/Right"],  # select fingertapping events, discard others
    before=5 * units.s,  # seconds before stimulus
    after=30 * units.s,  # seconds after stimulus
)

# calculate baseline
baseline = epochs.sel(reltime=(epochs.reltime < 0)).mean("reltime")

# subtract baseline
epochs_blcorrected = epochs - baseline

# group trials by trial_type. For each group individually average the epoch dimension
blockaverage = epochs_blcorrected.groupby("trial_type").mean("epoch")

## The TwoSurfaceHeadModel

Use the `get_standard_headmodel` function to construct a TSHM

In [ ]:
head = dot.get_standard_headmodel(HEAD_MODEL)

In [ ]:
head

In [ ]:
head.landmarks

Changing between coordinate systems:

In [ ]:
head_ras = head.apply_transform(head.t_ijk2ras)
head_ras


## Optode Registration
The optode coordinates from the recording must be aligned with the scalp surface. Currently, `cedaĺion` offers a simple registration method, which finds an affine transformation (scaling, rotating, translating) that matches the landmark positions of the head model and their digitized counter parts. Afterwards, optodes are snapped to the nearest vertex on the scalp.

In [ ]:
geo3d_snapped_ijk = head.align_and_snap_to_scalp(geo3d_meas)
display(geo3d_snapped_ijk)

## Light propagation in tissue

- construct forward model
- load Adot via `cedalion.datasets`
- logic for running MCX/Nirfaster is kept in the notebook to facilitate adaption to other heads

In [ ]:
fwm = dot.ForwardModel(head, geo3d_snapped_ijk, meas_list)

In [ ]:
if PRECOMPUTED_FLUENCE:
    if FORWARD_MODEL == "MCX":
        fluence_fname = cedalion.datasets.get_precomputed_fluence(DATASET, HEAD_MODEL)
    elif FORWARD_MODEL == "NIRFASTER":
        raise NotImplementedError(
            "Currently there are no precomputed NIRFASTER results available"
        )
else:
    fluence_fname = tmp_dir_path / "fluence.h5"

    if FORWARD_MODEL == "MCX":
        fwm.compute_fluence_mcx(fluence_fname)
    elif FORWARD_MODEL == "NIRFASTER":
        fwm.compute_fluence_nirfaster(fluence_fname)

### Calculate the sensitivity matrices
The forward model's function `compute_sensitivity` calculates the sensitivity matrix from the fluence file and saves the result in a new file.

In [ ]:
if PRECOMPUTED_FLUENCE:
    Adot = cedalion.datasets.get_precomputed_sensitivity(DATASET, HEAD_MODEL)
else:
    sensitivity_fname = tmp_dir_path / "sensitivity.h5"
    fwm.compute_sensitivity(fluence_fname, sensitivity_fname)
    Adot = load_Adot(sensitivity_fname)

The sensitivity matrix describes how an absorption change at a given surface vertex changes the optical density in a given channel and wavelength. 

The coordinate `is_brain` holds a mask to distinguish brain and scalp voxels.

In [ ]:
# load and display sensitivity matrix
display(Adot)

## Reconstruct the Image

### Gaussian Spatial Basis Functions

After constructing the SBFs once, the prepared GaussianSpatialBasisFunctions object can be stored in an hdf5 file.
If the file already exists, load it.

In [ ]:
# previous SFB implementation. uncomment for testing
#fname = tmp_dir_path / "sbf.h5"
#
#if fname.exists():
#    sbf = dot.OriginalGaussianSpatialBasisFunctions.from_file(fname)
#else:
#    sbf = dot.OriginalGaussianSpatialBasisFunctions(
#        head_ras,
#        Adot,
#        mask_threshold=-2,
#        threshold_brain=1 * units.mm,
#        threshold_scalp=5 * units.mm,
#        sigma_brain=1 * units.mm,
#        sigma_scalp=5 * units.mm,
#    )
#    sbf.to_file(fname)

# explicit parameters
sbf = dot.GaussianSpatialBasisFunctions(
    head_ras,
    Adot,
    mask_threshold=-2,
    threshold_brain=1 * units.mm,
    threshold_scalp=5 * units.mm,
    sigma_brain=1 * units.mm,
    sigma_scalp=5 * units.mm,
)
# alternatively: using predefined parameter sets
sbf = dot.GaussianSpatialBasisFunctions(head_ras, Adot, **dot.SBF_GAUSSIANS_DENSE)

### Test default parameter sets

In [ ]:
c_meas = measurement_variance(rec["od"].pint.dequantify(), calc_covariance=False)

recon = dot.ImageRecon(
    Adot,
    recon_mode="mua2conc",
    spatial_basis_functions=sbf,
    **dot.REG_TIKHONOV_ONLY,
)

recon_result = recon.reconstruct(blockaverage, c_meas)

recon = dot.ImageRecon(
    Adot,
    recon_mode="mua2conc",
    spatial_basis_functions=sbf,
    **dot.REG_PAPER_MUA_SBF
)

recon_result = recon.reconstruct(blockaverage, c_meas)

### Create combinations of ImageRecon parameters 

In [ ]:
timeseries = [
    # ('channel', 'wavelength', 'time')
    blockaverage.isel(reltime=[40], trial_type=0).rename({"reltime" : "time"}),
    # ('channel', 'wavelength')
    blockaverage.isel(reltime=40, trial_type=0),
    # ('trial_type', 'channel', 'wavelength', 'reltime')
    blockaverage.isel(reltime=[40], trial_type=[0]),
]

c_meas = measurement_variance(rec["od"].pint.dequantify(), calc_covariance=False)

c_meas_with_time = xr.concat([c_meas, c_meas, c_meas], dim="time").assign_coords(
    {"time": [0.0, 1.0, 2.0], "sample": ("time", [0, 1, 2])}
)

sbfs = [sbf, None]
recon_modes = ["mua", "conc", "mua2conc"]
brainonly_modes = [False, True]
alpha_meas_params = [0.01]
alpha_spatial_params = [None, 0.01]
c_meas_params = [None, c_meas, c_meas_with_time]

combinations = list(
    itertools.product(
        timeseries,
        sbfs,
        recon_modes,
        brainonly_modes,
        alpha_meas_params,
        alpha_spatial_params,
        c_meas_params,
    )
)
print(f"generated {len(combinations)} combinations")

In [ ]:
def run_comb(i, catch):
    ts, sbf_, recon_mode, brainonly_mode, alpha_meas, alpha_spatial, c_meas_ = combinations[i]

    print(
        f"{i}/{len(combinations)}: {' x '.join(ts.dims)} sbf_={sbf_ is not None} "
        f"{recon_mode=} {brainonly_mode=} {alpha_meas=} {alpha_spatial=} "
        f"c_meas_={c_meas_ is not None}"
    )

    # construct ImageRecon object
    recon = dot.ImageRecon(
        Adot,
        recon_mode=recon_mode,
        brain_only=brainonly_mode,
        alpha_meas=alpha_meas,
        alpha_spatial=alpha_spatial,
        apply_c_meas=(c_meas_ is not None),
        spatial_basis_functions=sbf_,
    )

    if catch:
        # run reconstruction, catch any exception
        try:
            recon_result = recon.reconstruct(ts, c_meas=c_meas_)
            if c_meas_ is not None:
                image_noise = recon.get_image_noise(c_meas=c_meas_)
            print(f"recon_result dims:{recon_result.dims} shape: {recon_result.shape}")
        except Exception:
            print("Error")
            print(traceback.format_exc())
            return None
        else:
            print("OK")
    else:
        # run reconstruction, don't catch any exception. jump into debugger
        recon_result = recon.reconstruct(ts, c_meas=c_meas)
        if c_meas_ is not None:
            image_noise = recon.get_image_noise(c_meas=c_meas_)
        print(f"recon_result dims:{recon_result.dims} shape: {recon_result.shape}")

    return recon_result

### Run a single combination of recon settings. Entry point for debugging

In [ ]:
run_comb(37, catch=False)

### Run all combinations. Catch and count combinations that throw errors.

In [ ]:
print(80*"-")
oks = 0
for i in range(len(combinations)):
    oks += int(run_comb(i, catch=True) is not None)
    print(80*"-")

print("="*80)
print(f"OK={oks} / {len(combinations)}")
print("="*80)


## Inspect Reconstruction results

In [ ]:
plot_combinations = [
    i
    for i, (
        ts,
        sbf_,
        recon_mode,
        brainonly_mode,
        alpha_meas,
        alpha_spatial,
        c_meas_,
    ) in enumerate(combinations)
    if (recon_mode == "conc")
    and (brainonly_mode == False)
    and (ts.dims == ("trial_type", "channel", "wavelength", "reltime"))
]
display(plot_combinations)

In [ ]:
for i_comb in plot_combinations:

    recon_result = run_comb(i_comb, catch=False)

    #display(recon_result)

    # reduce recon_result to vertex x chromo
    X_ts = recon_result.sel(vertex=recon_result.is_brain).isel(trial_type=0, reltime=0)

    image_recon_multi_view(
        X_ts,  # time series data; can be 2D (static) or 3D (dynamic)
        head,
        cmap="seismic",
        #clim=(-1e-6, 1e-6),
        view_type="hbr_brain",
        title_str="HbO / uM",
        # filename=filename_multiview,
        SAVE=False,
        # time_range=(-5,30,0.5)*units.s,
        # fps=6,
        geo3d_plot=None,  #  geo3d_plot
        wdw_size=(1024, 768),
    )

In [ ]:
Adot